# Training the Base Keyword-Spotting Classifier

Every attack and defense notebook in this repo targets a single classifier: a small convolutional
network over log-mel spectrograms trained to recognize 10 spoken commands (Google Speech Commands,
Warden 2018). This is the audio equivalent of `defenses/01_adversarial_training.ipynb`'s TrafficNet
model in the companion vision repo, trained from scratch rather than downloaded pretrained, so the
full attack-defense loop, including adversarial training, is actually possible on this codebase
instead of only attacking a frozen, external model.

**Task:** classify a 1-second, 16kHz utterance into one of 10 keywords: yes, no, up, down, left,
right, on, off, stop, go, the same 10-word subset used throughout the original TensorFlow Speech
Commands tutorials and benchmark papers.


In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import soundfile as sf
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)
DEVICE = 'cpu'
os.makedirs('speech_commands_data', exist_ok=True)  # SPEECHCOMMANDS(download=True) expects this to already exist

def _load_wav(path, frame_offset=0, num_frames=-1, normalize=True, channels_first=True, format=None, buffer_size=4096, backend=None):
    """Replaces torchaudio's TorchCodec-based loader (needs a separately installed FFmpeg, not
    available here) with soundfile, which reads WAV directly with no extra system dependencies."""
    data, sample_rate = sf.read(path, dtype='float32')
    waveform = torch.from_numpy(data)
    if waveform.dim() == 1:
        waveform = waveform.unsqueeze(1)
    if channels_first:
        waveform = waveform.transpose(0, 1)
    return waveform, sample_rate

torchaudio.load = _load_wav

### Setup: Dataset
`torchaudio.datasets.SPEECHCOMMANDS` downloads the official train/validation/test split lists
along with the audio. We keep only the 10 command classes above (the dataset also ships filler
words and background noise clips used for the harder 35-class task, out of scope here) and cap
the number of clips per class, the audio equivalent of the reduced-scale GTSRB training set used
for `defenses/01_adversarial_training.ipynb` in the vision repo, to keep training tractable on CPU.

In [2]:
LABELS = ['yes', 'no', 'up', 'down', 'left', 'right', 'on', 'off', 'stop', 'go']
LABEL_TO_IDX = {label: i for i, label in enumerate(LABELS)}
SAMPLE_RATE = 16000

MAX_PER_CLASS = {'training': 400, 'validation': 60, 'testing': 60}

class KeywordSubset(Dataset):
    """Wraps SPEECHCOMMANDS, keeping only the 10 target labels and capping clips per class."""
    def __init__(self, subset):
        full = torchaudio.datasets.SPEECHCOMMANDS(root='speech_commands_data', download=True, subset=subset)
        limit = MAX_PER_CLASS[subset]
        counts = {label: 0 for label in LABELS}
        self.items = []
        # get_metadata() reads only the filename/label, not the audio, so filtering the ~85k-file
        # training split down to the ~4000 clips actually needed here doesn't decode every waveform
        for i in range(len(full)):
            _, _, label, *_ = full.get_metadata(i)
            if label not in LABEL_TO_IDX or counts[label] >= limit:
                continue
            counts[label] += 1
            waveform, sr, label, *_ = full[i]
            self.items.append((waveform, label))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        waveform, label = self.items[idx]
        # Pad or trim every clip to exactly 1 second so batches can be stacked
        if waveform.shape[1] < SAMPLE_RATE:
            waveform = F.pad(waveform, (0, SAMPLE_RATE - waveform.shape[1]))
        else:
            waveform = waveform[:, :SAMPLE_RATE]
        return waveform, LABEL_TO_IDX[label]

train_set = KeywordSubset('training')
val_set = KeywordSubset('validation')
test_set = KeywordSubset('testing')
print(f"Training: {len(train_set)}, Validation: {len(val_set)}, Testing: {len(test_set)} clips")

Training: 4000, Validation: 600, Testing: 600 clips


### Setup: Model
Raw waveforms go through a log-mel spectrogram front end (64 mel bands), then a small 4-layer
CNN, the audio analogue of the compact CNN used for TrafficNet in the vision repo: a handful of
conv/pool blocks feeding a linear classifier, not a huge pretrained backbone.

In [3]:
mel_spectrogram = torchaudio.transforms.MelSpectrogram(sample_rate=SAMPLE_RATE, n_mels=64, n_fft=400, hop_length=160)
to_db = torchaudio.transforms.AmplitudeToDB()

def waveform_to_spectrogram(waveform):
    """(batch, 1, 16000) waveform -> (batch, 1, 64, ~101) log-mel spectrogram, ready for a 2D CNN."""
    return to_db(mel_spectrogram(waveform))

class KeywordCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(2)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, spectrogram):
        x = self.pool(F.relu(self.bn1(self.conv1(spectrogram))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.global_pool(x).flatten(1)
        return self.fc(x)

model = KeywordCNN(num_classes=len(LABELS)).to(DEVICE)
print(model)

KeywordCNN(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (global_pool): AdaptiveAvgPool2d(output_size=1)
  (fc): Linear(in_features=64, out_features=10, bias=True)
)


### Training
Plain supervised cross-entropy training, Adam, a handful of epochs since the model is small and
the training set has been deliberately kept compact.

In [4]:
BATCH_SIZE = 32
EPOCHS = 25

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def evaluate(loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for waveforms, labels in loader:
            logits = model(waveform_to_spectrogram(waveforms))
            preds = torch.argmax(logits, dim=-1)
            correct += int((preds == labels).sum())
            total += len(labels)
    return correct / total

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for waveforms, labels in train_loader:
        optimizer.zero_grad()
        logits = model(waveform_to_spectrogram(waveforms))
        loss = F.cross_entropy(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * len(labels)

    train_loss = total_loss / len(train_set)
    val_acc = evaluate(val_loader)
    print(f"Epoch {epoch + 1}/{EPOCHS}, train loss: {train_loss:.4f}, val accuracy: {val_acc * 100:.1f}%")

C:\Users\frang\AppData\Local\Temp\ipykernel_26700\2955397033.py:29: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:821.)
  total_loss += float(loss) * len(labels)


Epoch 1/25, train loss: 2.2057, val accuracy: 21.5%


Epoch 2/25, train loss: 2.0369, val accuracy: 23.5%


Epoch 3/25, train loss: 1.9226, val accuracy: 21.2%


Epoch 4/25, train loss: 1.8030, val accuracy: 23.0%


...

Epoch 21/25, train loss: 0.8615, val accuracy: 58.3%


Epoch 22/25, train loss: 0.8604, val accuracy: 48.2%


Epoch 23/25, train loss: 0.8043, val accuracy: 49.3%


Epoch 24/25, train loss: 0.7875, val accuracy: 31.2%


Epoch 25/25, train loss: 0.7771, val accuracy: 48.8%


### Test Accuracy and Saving
The saved weights (`models/keyword_spotter.pth`) are the target model for every attack and
defense notebook in this repo, loaded via the same `KeywordCNN` class and `waveform_to_spectrogram`
front end defined above.

In [5]:
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False)
test_acc = evaluate(test_loader)
print(f"Test accuracy: {test_acc * 100:.1f}%")

torch.save(model.state_dict(), 'keyword_spotter.pth')
print("Saved models/keyword_spotter.pth")

Test accuracy: 58.2%
Saved models/keyword_spotter.pth
